# 07 — Carga de datos a MySQL
### TFM: Análisis multidimensional del estrés estudiantil

```
================================================================================
07 - CARGA DE DATOS A MySQL (modelo v2 con dimensiones conformes)
================================================================================
Carga los 12 CSV en las tablas de hechos de tfm_estres_estudiantil_2,
calculando en cada fila sus claves conformes (id_genero, id_rango_edad,
id_nivel_educativo, id_nivel_estres) con los mismos criterios del script 06.

REQUISITOS (una sola vez, en consola/Anaconda Prompt):
    pip install sqlalchemy pymysql

CONFIGURACION XAMPP por defecto: usuario root sin contrasena, host localhost.
Ejecutar DESPUES de modelo_datos_mysql_v2.sql y esquema_estrella_powerbi.sql.

NOTA b4_dass_item: son ~1,67 millones de filas (42 items x 39.767 personas).
Por defecto NO se carga (CARGAR_DASS_ITEM = False); activar solo si se
necesita el detalle por item, y prever varios minutos de carga.
================================================================================
```

---
## ⚠️ Antes de ejecutar: qué va en cada herramienta

Este proceso tiene **dos mitades** que se ejecutan en sitios distintos:

| Paso | Dónde | Archivo | Estado |
|---|---|---|---|
| 1. Crear la base y las 19 tablas | phpMyAdmin (pestaña Importar) | `modelo_datos_mysql_v2.sql` | ✅ ya hecho |
| 2. Crear las 6 vistas de la estrella | phpMyAdmin | `esquema_estrella_powerbi.sql` | ✅ ya hecho |
| 3. **Cargar los datos de los CSV** | **este notebook** | — | ⬅️ estás aquí |
| 4. Conectar los dashboards | Power BI | vistas `vw_*` | pendiente |

**Jupyter no reemplaza a phpMyAdmin**: el notebook se *conecta* al MySQL de XAMPP por red local (localhost) usando `sqlalchemy` + `pymysql`, lee los CSV con pandas y hace los INSERT. Los scripts `.sql` siguen yendo en phpMyAdmin porque son sentencias de definición (crear tablas/vistas).

**Requisito imprescindible:** el panel de control de XAMPP debe tener **MySQL en verde (Running)** antes de ejecutar nada de aquí.

### 🚫 Regla de oro de este notebook
La carga usa modo `append`. **Ejecuta las celdas UNA sola vez y en orden.** Si re-ejecutas una celda de carga, duplicarás esas filas. La celda de verificación (más abajo) comprueba que las tablas estén vacías antes de empezar y te avisa si no lo están.

Si necesitas reintentar desde cero: vuelve a importar en phpMyAdmin `modelo_datos_mysql_v2.sql` (hace DROP y reconstruye vacío) + `esquema_estrella_powerbi.sql`, y entonces relanza este notebook.


### Paso 0 — Instalar conectores

In [ ]:
# Instalar los conectores (solo la primera vez; si ya los tienes, no hace nada)
%pip install sqlalchemy pymysql --quiet
print("Conectores listos. Reinicia el kernel si es la primera instalacion.")

### Paso 1 — Configuración y conexión
Ajusta `DB_PASS` si tu MySQL tiene contraseña. `CARGAR_DASS_ITEM` queda en `False`: esa tabla son ~1,67 millones de filas y no la necesitan los dashboards.

In [ ]:
import os
import pandas as pd
from sqlalchemy import create_engine, text

# ── Configuracion global ──────────────────────────────────────────────────────
BASE = r"C:\Users\nicop\OneDrive\Documentos\Maestria\INESDI & UNIE\BA & IA\TFM\Propuesta"

DB_USER, DB_PASS = "root", ""            # credenciales XAMPP por defecto
DB_HOST, DB_NAME = "localhost", "tfm_estres_estudiantil_2"
CARGAR_DASS_ITEM = False                  # ver nota de cabecera

engine = create_engine(
    f"mysql+pymysql://{DB_USER}:{DB_PASS}@{DB_HOST}/{DB_NAME}?charset=utf8mb4")

# ── Claves de las dimensiones conformes (segun INSERTs del modelo v2) ─────────
ID_GENERO = {"NR": 0, "M": 1, "F": 2, "O": 3}
ID_RANGO  = {"No reportado": 0, "NR": 0, "<18": 1, "18-25": 2,
             "26-35": 3, "36-50": 4, ">50": 5}
ID_EDU    = {"NR": 0, "No reportado": 0, "Secundaria o menos": 1,
             "Grado universitario": 2, "Posgrado": 3}
ID_ESTRES = {"No aplica": 0, "Bajo": 1, "Moderado": 2, "Alto": 3}
ID_SUBRED = {n: i for i, n in enumerate(
    ["ptsd", "relationships", "anxiety", "domesticviolence", "assistance",
     "survivorsofabuse", "homeless", "almosthomeless", "stress", "food_pantry"], 1)}

### Paso 2 — Comprobación previa (evita duplicados)
Si alguna tabla ya tiene filas, esta celda **detiene** el proceso.

In [ ]:
# VERIFICACION DE SEGURIDAD: las tablas deben estar vacias antes de cargar
from sqlalchemy import inspect

tablas_hechos = ["b1_academic_stress", "b1_anxiety", "b1_mental_health_survey",
                 "b1_student_habits", "b2_social_media_sleep", "b2_smmh",
                 "b2_lifestyle", "b3_dreaddit_post", "b3_depression_student",
                 "b4_dass_respondente", "b4_mental_health"]

with engine.connect() as con:
    print("Conexion OK ->", DB_NAME, "\n")
    ocupadas = []
    for t in tablas_hechos:
        n = con.execute(text(f"SELECT COUNT(*) FROM {t}")).scalar()
        print(f"  {t:<26} {n:>8} filas")
        if n > 0:
            ocupadas.append(t)

if ocupadas:
    raise SystemExit(
        "\n*** DETENIDO: estas tablas YA tienen datos: " + ", ".join(ocupadas) +
        "\n*** Si vuelves a cargar, duplicaras las filas."
        "\n*** Para recargar limpio: reimporta modelo_datos_mysql_v2.sql y "
        "esquema_estrella_powerbi.sql en phpMyAdmin, y vuelve a ejecutar este notebook.")
else:
    print("\nTodas las tablas vacias: se puede proceder con la carga.")

### Paso 3 — Funciones de armonización
Mismos criterios que el notebook 06: traducen género, edad, educación y la variable objetivo nativa a las claves de las dimensiones conformes.

In [ ]:
# ── Mapeadores (identicos al script 06) ───────────────────────────────────────
def map_genero(v):
    if pd.isna(v): return "NR"
    s = str(v).strip().lower()
    if s in ("male", "m", "hombre", "1", "1.0"):   return "M"
    if s in ("female", "f", "mujer", "2", "2.0"):  return "F"
    if s in ("", "nan", "prefer not to say"):      return "NR"
    return "O"

def map_rango_edad(v):
    try: e = float(v)
    except (TypeError, ValueError): return "NR"
    if pd.isna(e) or e < 13 or e > 90: return "NR"
    if e < 18:  return "<18"
    if e <= 25: return "18-25"
    if e <= 35: return "26-35"
    if e <= 50: return "36-50"
    return ">50"

def likert_a_nivel(v):
    try: x = float(v)
    except (TypeError, ValueError): return "No aplica"
    if x <= 2: return "Bajo"
    if x <= 3: return "Moderado"
    return "Alto"

def edad_o_none(v):
    """Edad valida 13-90 o None (para columnas TINYINT con CHECK)."""
    try: e = float(v)
    except (TypeError, ValueError): return None
    return int(e) if 13 <= e <= 90 else None

def conformes(df, genero=None, edad=None, edu=None, estres=None):
    """Devuelve las 4 columnas de claves conformes para el DataFrame dado."""
    out = pd.DataFrame(index=df.index)
    out["id_genero"] = (genero.map(map_genero) if genero is not None
                        else "NR")
    out["id_genero"] = out["id_genero"].map(ID_GENERO)
    out["id_rango_edad"] = (edad.map(map_rango_edad) if edad is not None
                            else "NR")
    out["id_rango_edad"] = out["id_rango_edad"].map(ID_RANGO)
    out["id_nivel_educativo"] = (edu if edu is not None else "NR")
    out["id_nivel_educativo"] = out["id_nivel_educativo"].fillna("NR").map(ID_EDU).fillna(0).astype(int)
    out["id_nivel_estres"] = estres.map(ID_ESTRES).fillna(0).astype(int)
    return out

def cargar(nombre_tabla, df):
    df.to_sql(nombre_tabla, engine, if_exists="append", index=False,
              chunksize=2000)
    print(f"  {nombre_tabla:<26} {len(df):>7} filas cargadas")

print("=" * 80)
print("CARGA DE DATOS - modelo v2 con dimensiones conformes")
print("=" * 80)

---
## Carga por dataset
Ejecuta las celdas **en orden y una sola vez**. Cada una imprime las filas insertadas.

**B1.1 Academic Stress** (140 filas). Sin género ni edad en el origen → claves conformes a *No reportado*; la etapa académica se aprovecha como nivel educativo.

In [ ]:
df = pd.read_csv(os.path.join(BASE, "Bloque 1",
    "1 Student Academic Stress Real World Dataset",
    "academic Stress level - maintainance 1.csv"))
df.columns = [c.strip() for c in df.columns]
T = "Rate your academic stress index"
edu_stage = df["Your Academic Stage"].map({
    "undergraduate": "Grado universitario", "post-graduate": "Posgrado",
    "high school": "Secundaria o menos"})
out = conformes(df, edu=edu_stage, estres=df[T].map(likert_a_nivel))
out["id_dataset"] = 1
out["ts_respuesta"] = df["Timestamp"].astype(str).str[:30]
out["etapa_academica"] = df["Your Academic Stage"]
out["presion_pares"] = df["Peer pressure"]
out["presion_hogar"] = df["Academic pressure from your home"]
out["entorno_estudio"] = df["Study Environment"]
out["estrategia_afrontamiento"] = df["What coping strategy you use as a student?"].astype(str).str[:120]
out["habitos_nocivos"] = df["Do you have any bad habits like smoking, drinking on a daily basis?"].astype(str).str[:30]
out["competencia_academica"] = df[[c for c in df.columns if "competition" in c][0]]
out["indice_estres"] = df[T]
cargar("b1_academic_stress", out)

**B1.2 Anxiety GAD-7** (13.464 filas). Nivel de estrés por umbrales clínicos GAD-7 (0-4 / 5-9 / ≥10). Lectura con `encoding='latin1'`.

In [ ]:
df = pd.read_csv(os.path.join(BASE, "Bloque 1", "1 Student anxiety dataset",
                              "anxiety.csv"), encoding="latin1")
def gad_a_nivel(v):
    try: x = float(v)
    except (TypeError, ValueError): return "No aplica"
    if x <= 4: return "Bajo"
    if x <= 9: return "Moderado"
    return "Alto"
edu_anx = df["Degree"].map({
    "Bachelor\xa0(or equivalent)": "Grado universitario",
    "Master\xa0(or equivalent)": "Posgrado",
    "Ph.D., Psy. D., MD\xa0(or equivalent)": "Posgrado",
    "High school diploma (or equivalent)": "Secundaria o menos",
    "None": "Secundaria o menos"})
out = conformes(df, genero=df["Gender"], edad=df["Age"], edu=edu_anx,
                estres=df["GAD_T"].map(gad_a_nivel))
out["id_dataset"] = 2
out["num_secuencial_csv"] = df["S. No."]
for i in range(1, 8):
    out[f"gad{i}"] = df[f"GAD{i}"]
out["gade"] = df["GADE"].astype(str).str[:60]
out["swl_total"] = df["SWL_T"]
out["spin_total"] = df["SPIN_T"]
out["gad_total"] = df["GAD_T"]
out["horas_juego"] = pd.to_numeric(df["Hours"], errors="coerce").clip(upper=9999)
out["edad"] = df["Age"].map(edad_o_none)
out["ocupacion"] = df["Work"].astype(str).str[:60]
out["pais_residencia"] = df["Residence_ISO3"].astype(str).str[:3]
cargar("b1_anxiety", out)

**B1.3 Mental Health Survey** (87 filas). Ojo: la columna `academic_workload` trae un espacio final en el CSV — se limpia con `strip()` al cargar.

In [ ]:
df = pd.read_csv(os.path.join(BASE, "Bloque 1",
                              "1 Student Mental Health Survey/MentalHealthSurvey.csv"))
df.columns = [c.strip() for c in df.columns]   # 'academic_workload ' trae espacio final
edu_ms = df["degree_level"].map({"Undergraduate": "Grado universitario",
                                  "Postgraduate": "Posgrado"})
out = conformes(df, genero=df["gender"], edad=df["age"], edu=edu_ms,
                estres=df["depression"].map(likert_a_nivel))
out["id_dataset"] = 3
out["edad"] = df["age"].map(edad_o_none)
out["universidad"] = df["university"].astype(str).str[:60]
out["carrera"] = df["degree_major"].astype(str).str[:60]
out["anio_academico"] = df["academic_year"].astype(str).str[:20]
out["cgpa_rango"] = df["cgpa"].astype(str).str[:20]
out["satisfaccion_estudios"] = df["study_satisfaction"]
out["carga_academica"] = df["academic_workload"]
out["presion_academica"] = df["academic_pressure"]
out["aislamiento"] = df["isolation"]
out["inseguridad_futuro"] = df["future_insecurity"]
out["depresion"] = df["depression"]
out["ansiedad"] = df["anxiety"]
cargar("b1_mental_health_survey", out)

**B1.4 Student Habits** (1.000 filas). Dataset sintético: se carga para la capa descriptiva. `mental_health_rating` está invertida (más alto = mejor), por eso el mapeo también se invierte.

In [ ]:
df = pd.read_csv(os.path.join(BASE, "Bloque 1",
    "1 Student-Habits-vs-Academic-Performance", "student_habits_performance.csv"))
def salud_inv(v):
    try: x = float(v)
    except (TypeError, ValueError): return "No aplica"
    if x >= 8: return "Bajo"
    if x >= 5: return "Moderado"
    return "Alto"
out = conformes(df, genero=df["gender"], edad=df["age"],
                estres=df["mental_health_rating"].map(salud_inv))
out["id_dataset"] = 4
out["id_estudiante"] = df["student_id"]
out["edad"] = df["age"].map(edad_o_none)
out["horas_estudio_dia"] = df["study_hours_per_day"]
out["horas_redes"] = df["social_media_hours"]
out["horas_streaming"] = df["netflix_hours"]
out["trabajo_parcial"] = df["part_time_job"]
out["asistencia_pct"] = df["attendance_percentage"]
out["horas_sueno"] = df["sleep_hours"]
out["calidad_dieta"] = df["diet_quality"].astype(str).str[:20]
out["frecuencia_ejercicio"] = df["exercise_frequency"]
out["educacion_padres"] = df["parental_education_level"].astype("string").str[:30]
out["calidad_internet"] = df["internet_quality"].astype(str).str[:20]
out["salud_mental"] = df["mental_health_rating"]
out["nota_examen"] = df["exam_score"].clip(0, 100)
cargar("b1_student_habits", out)

**B2.1 Social Media/Sleep** (11.000 filas). Sintético. Sin columna de género en el origen → *No reportado*.

In [ ]:
df = pd.read_csv(os.path.join(BASE, "Bloque 2",
    "2 Social Media Usage vs Sleep, Stress & Productivity",
    "social_media_sleep_stress_productivity_11000.csv"))
out = conformes(df, edad=df["age"],
                estres=df["stress_level"].map({"Low": "Bajo", "Medium": "Moderado",
                                                "High": "Alto"}))
out["id_dataset"] = 5
out["id_usuario_red"] = df["user_id"]
out["edad"] = df["age"].map(edad_o_none)
out["plataforma"] = df["platform"].astype(str).str[:30]
out["horas_redes"] = df["social_media_hours"]
out["horas_pantalla"] = df["daily_screen_time_hours"].clip(0, 24)
out["horas_sueno"] = df["sleep_hours"].clip(0, 24)
out["horas_estudio_trabajo"] = df["study_work_hours"]
out["minutos_ejercicio"] = df["exercise_minutes"]
out["productividad"] = df["productivity_score"]
out["nivel_estres_nativo"] = df["stress_level"]
cargar("b2_social_media_sleep", out)

**B2.2 SMMH** (481 filas). Las columnas se acceden por posición porque sus nombres son preguntas completas de la encuesta.

In [ ]:
df = pd.read_csv(os.path.join(BASE, "Bloque 2",
                              "2 Social Media and Mental Health/smmh.csv"))
c = list(df.columns)
lik = lambda i: pd.to_numeric(df[c[i]], errors="coerce")
out = conformes(df, genero=df[c[2]], edad=df[c[1]],
                estres=lik(18).map(likert_a_nivel))
out["id_dataset"] = 6
out["ts_respuesta"] = df[c[0]].astype(str).str[:30]
out["edad"] = df[c[1]].map(edad_o_none)
out["estado_relacion"] = df[c[3]].astype(str).str[:30]
out["ocupacion"] = df[c[4]].astype(str).str[:30]
out["usa_redes"] = df[c[6]].map(lambda v: "Yes" if str(v).lower().startswith("y") else "No")
out["plataformas"] = df[c[7]].astype(str).str[:255]
out["tiempo_diario"] = df[c[8]].astype(str).str[:40]
campos = ["uso_sin_proposito", "distraccion_redes", "inquietud_sin_redes",
          "distraccion_facil", "preocupacion", "dificultad_concentracion",
          "comparacion_social", "sentimiento_comparacion", "busqueda_validacion",
          "se_siente_deprimido", "fluctuacion_interes", "problemas_sueno"]
for campo, idx in zip(campos, range(9, 21)):
    out[campo] = lik(idx)
cargar("b2_smmh", out)

**B2.3 Lifestyle** (3.000 filas). Sintético. Los NaN de condición de salud mental se recodifican como 'Ninguna'.

In [ ]:
df = pd.read_csv(os.path.join(BASE, "Bloque 2",
    "2 Mental_Health_Lifestyle GITHUB/Mental_Health_Lifestyle_Dataset.csv"))
out = conformes(df, genero=df["Gender"], edad=df["Age"],
                estres=df["Stress Level"].map({"Low": "Bajo", "Moderate": "Moderado",
                                                "High": "Alto"}))
out["id_dataset"] = 7
out["pais"] = df["Country"].astype(str).str[:30]
out["edad"] = df["Age"].map(edad_o_none)
out["nivel_ejercicio"] = df["Exercise Level"].astype(str).str[:20]
out["tipo_dieta"] = df["Diet Type"].astype(str).str[:30]
out["horas_sueno"] = df["Sleep Hours"].clip(0, 24)
out["nivel_estres_nativo"] = df["Stress Level"]
out["condicion_salud_mental"] = df["Mental Health Condition"].fillna("Ninguna").astype(str).str[:30]
out["horas_trabajo_semana"] = df["Work Hours per Week"].clip(0, 99)
out["horas_pantalla"] = df["Screen Time per Day (Hours)"]
out["interaccion_social"] = df["Social Interaction Score"]
out["felicidad"] = df["Happiness Score"]
cargar("b2_lifestyle", out)

**B3.1 Dreaddit train + test** (2.834 + 715 filas). Aplica la regla de calidad R7: excluye los 4 fragmentos corruptos con texto `#NAME?`. Ambas particiones van a la misma tabla diferenciadas por el campo `particion`.

In [ ]:
for part, ds_id in [("train", 8), ("test", 9)]:
    df = pd.read_csv(os.path.join(BASE, "Bloque 3",
        "3 Stress Analysis in Social Media Dataset", f"dreaddit-{part}.csv"))
    df = df[~df["text"].str.strip().isin(["#NAME?", "#REF!", "#VALUE!"])]  # R7
    out = conformes(df, estres=df["label"].map({0: "Bajo", 1: "Alto"}))
    out["id_dataset"] = ds_id
    out["id_fragmento_post"] = df["id"]
    out["id_subreddit"] = df["subreddit"].map(ID_SUBRED)
    out["particion"] = part
    out["post_id_origen"] = df["post_id"].astype(str).str[:20]
    out["rango_frases"] = df["sentence_range"].astype(str).str[:20]
    out["texto"] = df["text"]
    out["etiqueta_estres"] = df["label"]
    out["confianza_etiqueta"] = df["confidence"]
    out["ts_publicacion"] = pd.to_numeric(df["social_timestamp"], errors="coerce")
    out["karma"] = pd.to_numeric(df["social_karma"], errors="coerce")
    out["num_comentarios"] = pd.to_numeric(df["social_num_comments"], errors="coerce")
    out["sentimiento"] = df["sentiment"]
    out["liwc_ansiedad"] = df["lex_liwc_anx"]
    out["liwc_emo_negativa"] = df["lex_liwc_negemo"]
    out["liwc_emo_positiva"] = df["lex_liwc_posemo"]
    out["liwc_primera_persona"] = df["lex_liwc_i"]
    out["legibilidad_fk"] = df["syntax_fk_grade"]
    cargar("b3_dreaddit_post", out)

**B3.2 Depression Student** (502 filas).

In [ ]:
df = pd.read_csv(os.path.join(BASE, "Bloque 3",
    "3 Student_Mental_Health_Analysis GITHUB/Depression+Student+Dataset.csv"))
out = conformes(df, genero=df["Gender"], edad=df["Age"],
                estres=df["Depression"].map({"No": "Bajo", "Yes": "Alto"}))
out["id_dataset"] = 10
out["edad"] = df["Age"].map(edad_o_none)
out["presion_academica"] = df["Academic Pressure"]
out["satisfaccion_estudios"] = df["Study Satisfaction"]
out["duracion_sueno"] = df["Sleep Duration"].astype(str).str[:30]
out["habitos_alimentarios"] = df["Dietary Habits"].astype(str).str[:30]
out["pensamientos_suicidas"] = df["Have you ever had suicidal thoughts ?"]
out["horas_estudio"] = df["Study Hours"]
out["estres_financiero"] = df["Financial Stress"]
out["antecedentes_familiares"] = df["Family History of Mental Illness"]
out["depresion"] = df["Depression"]
cargar("b3_depression_student", out)

**B4.1 DASS-42** (39.767 filas tras filtrar edades inválidas). Calcula las tres subescalas con la clave oficial del codebook y aplica los baremos DASS para el nivel conforme. La carga opcional de `b4_dass_item` (1,67 M de filas) está desactivada por defecto.

In [ ]:
df = pd.read_csv(os.path.join(BASE, "Bloque 4",
    "4 Depression Anxiety Stress Scales Responses/data.csv"),
    sep="\t", low_memory=False)
df = df[(df["age"] >= 13) & (df["age"] <= 90)].reset_index(drop=True)
dep_i = [3,5,10,13,16,17,21,24,26,31,34,37,38,42]
anx_i = [2,4,7,9,15,19,20,23,25,28,30,36,40,41]
str_i = [1,6,8,11,12,14,18,22,27,29,32,33,35,39]
sub = lambda items: df[[f"Q{i}A" for i in items]].sum(axis=1)
df["dep_s"], df["anx_s"], df["str_s"] = sub(dep_i), sub(anx_i), sub(str_i)
def dass_nivel(v):
    if v <= 28: return "Bajo"
    if v <= 39: return "Moderado"
    return "Alto"
edu_dass = df["education"].map({1: "Secundaria o menos", 2: "Secundaria o menos",
                                 3: "Grado universitario", 4: "Posgrado"})
out = conformes(df, genero=df["gender"], edad=df["age"], edu=edu_dass,
                estres=df["str_s"].map(dass_nivel))
out["id_dataset"] = 11
out["pais"] = df["country"].astype(str).str[:2]
out["edad"] = df["age"].astype(int)
out["ingles_nativo"] = df["engnat"]
out["religion"] = df["religion"]
out["orientacion"] = df["orientation"]
out["estado_civil"] = df["married"]
out["carrera"] = df["major"].astype("string").str[:120]
out["seg_intro"] = df["introelapse"].clip(0, 4_000_000_000)
out["seg_test"] = df["testelapse"].clip(0, 4_000_000_000)
out["seg_encuesta"] = df["surveyelapse"].clip(0, 4_000_000_000)
out["validez_vcl_ok"] = (df[["VCL6", "VCL9", "VCL12"]].sum(axis=1) == 0)
out["score_depresion"] = df["dep_s"]
out["score_ansiedad"] = df["anx_s"]
out["score_estres"] = df["str_s"]
cargar("b4_dass_respondente", out)

# --------- b4_dass_item (opcional, ~1,67M filas) ------------------------------
if CARGAR_DASS_ITEM:
    print("  Cargando b4_dass_item (esto tarda varios minutos)...")
    with engine.begin() as con:
        ids = pd.read_sql("SELECT id_respondente_dass FROM b4_dass_respondente "
                          "ORDER BY id_respondente_dass", con)
    df = df.reset_index(drop=True)
    df["id_resp"] = ids["id_respondente_dass"].values
    sub_map = ({i: "depresion" for i in dep_i} | {i: "ansiedad" for i in anx_i}
               | {i: "estres" for i in str_i})
    bloques = []
    for i in range(1, 43):
        b = pd.DataFrame({
            "id_respondente_dass": df["id_resp"],
            "numero_item": i,
            "subescala": sub_map[i],
            "respuesta": df[f"Q{i}A"],
            "posicion_en_test": df[f"Q{i}I"],
            "ms_respuesta": pd.to_numeric(df[f"Q{i}E"], errors="coerce")
                              .clip(0, 4_000_000_000),
        })
        bloques.append(b)
    items = pd.concat(bloques, ignore_index=True)
    items = items[items["respuesta"].between(1, 4)]
    cargar("b4_dass_item", items)

**B4.2 Mental Health** (261.328 filas). Es el dataset más grande: esta celda tarda unos minutos. Sin edad en el origen → *No reportado*.

In [ ]:
df = pd.read_csv(os.path.join(BASE, "Bloque 4",
                              "4 Mental Health Dataset/Mental Health dataset1.csv"))
out = conformes(df, genero=df["Gender"],
                estres=df["IncreasingStress"].map({"No": "Bajo", "Maybe": "Moderado",
                                                    "Yes": "Alto"}))
out["id_dataset"] = 12
out["pais"] = df["Country"].astype(str).str[:40]
out["ocupacion"] = df["Occupation"].astype(str).str[:30]
out["autonomo"] = df["SelfEmployed"].astype("string")
out["antecedentes_familiares"] = df["FamilyHistory"]
out["tratamiento"] = df["Treatment"]
out["dias_en_casa"] = df["DaysIndoors"].astype(str).str[:30]
out["cambio_habitos"] = df["HabitsChange"].astype(str).str[:10]
out["historial_salud_mental"] = df["MentalHealthHistory"].astype(str).str[:10]
out["estres_creciente"] = df["IncreasingStress"].astype(str).str[:10]
out["cambios_humor"] = df["MoodSwings"].astype(str).str[:10]
out["debilidad_social"] = df["SocialWeakness"].astype(str).str[:10]
out["dificultad_afrontamiento"] = df["CopingStruggles"].astype(str).str[:10]
out["interes_trabajo"] = df["WorkInterest"].astype(str).str[:10]
out["entrevista_salud_mental"] = df["MentalHealthInterview"].astype(str).str[:10]
out["conoce_opciones"] = df["CareOptions"].astype(str).str[:15]
cargar("b4_mental_health", out)

# ── Verificacion final: filtro global del tutor sobre la estrella ─────────────
print("\n" + "=" * 80)
print("VERIFICACION: Mujeres 18-25 con estres Alto (via vw_hechos_estres)")
print("=" * 80)
q = """
SELECT ds.bloque, ds.dataset, COUNT(*) AS casos
FROM vw_hechos_estres h
JOIN vw_dim_dataset      ds ON ds.id_dataset     = h.id_dataset
JOIN vw_dim_genero       g  ON g.id_genero       = h.id_genero
JOIN vw_dim_rango_edad   r  ON r.id_rango_edad   = h.id_rango_edad
JOIN vw_dim_nivel_estres e  ON e.id_nivel_estres = h.id_nivel_estres
WHERE g.codigo='F' AND r.rango='18-25' AND e.nivel='Alto'
GROUP BY ds.bloque, ds.dataset ORDER BY casos DESC
"""
print(pd.read_sql(text(q), engine).to_string(index=False))
print("\nCARGA COMPLETADA.")

---
## ✅ Siguiente paso: Power BI

Con los datos cargados, en Power BI Desktop:

1. **Obtener datos → Base de datos MySQL** → Servidor: `localhost` · Base de datos: el nombre que uses (`tfm_estres_estudiantil_2`)
2. Selecciona **solo las 6 vistas `vw_*`** (no las tablas físicas)
3. En *Vista de modelo*, crea las relaciones **1:\*** desde cada dimensión hacia `vw_hechos_estres`, con dirección de filtro **única**
4. Medidas de arranque:
   ```
   Registros = COUNTROWS(vw_hechos_estres)
   % Estrés Alto = DIVIDE(CALCULATE([Registros], vw_dim_nivel_estres[nivel]="Alto"), [Registros])
   ```

Si Power BI pide el conector MySQL, instala *MySQL Connector/NET* desde la web de MySQL y reinicia Power BI.
